# Stochastic polarity from an initial condition

Compare $\alpha=2.5$ min⁻¹ (teal) and $\alpha=0.5$ min⁻¹ (blue).
All eight killers start with the dimensionless polarity $\boldsymbol{\mu}(0)=(1,\,0.5)$.
The killer noise settings match the migration notebook: polarity noise 0.75 and
translational noise 5.0. Polarity follows the stochastic equation

$$d\boldsymbol{\mu}=-\alpha\boldsymbol{\mu}\,dt+\sigma_\mu\,d\mathbf W.$$

Its conditional ensemble mean relaxes exponentially:

$$\mathbb{E}[\boldsymbol{\mu}(t)\mid\boldsymbol{\mu}(0)]
=\boldsymbol{\mu}(0)e^{-\alpha t}.$$

Individual cells keep fluctuating after the mean approaches zero. The decay time
$\tau=1/\alpha$ is 0.4 min and 2 min, respectively. The figure shows all eight
histories and the theoretical mean. A sample average over eight cells also fluctuates.
Time is interpreted in minutes; polarity is dimensionless. These settings illustrate
model behaviour and are not fitted to experimental trajectories.


In [ ]:
from dataclasses import replace
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display
import scienceplots

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "src/mantishrimp/simulation.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the MantiShrimp checkout.")
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
import mantishrimp
from mantishrimp import (
    Simulation, SimulationConfig, DomainConfig, MotilityConfig,
    MechanicsConfig, CytotoxicityConfig, OutputConfig,
)
if Path(mantishrimp.__file__).resolve() != SRC / "mantishrimp/__init__.py":
    raise RuntimeError("Restart the kernel to load this checkout's simulator.")

plt.style.use(["science", "no-latex"])
plt.rcParams.update({
    "font.family": "serif", "font.serif": ["STIXGeneral", "DejaVu Serif"],
    "mathtext.fontset": "stix", "text.usetex": False,
    "xtick.minor.visible": False, "ytick.minor.visible": False,
    "xtick.top": False, "ytick.right": False,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 1.1, "legend.frameon": False,
    "svg.fonttype": "none", "pdf.fonttype": 42,
})
FIGURE_DIR = PROJECT_ROOT / "paper/section_2/migration/figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


### 1. Settings

Edit `INITIAL_POLARITY` to change the starting vector. `CONDITIONS` sets the two
decay rates and their colours. The smaller time step resolves the fast decay.


In [ ]:
INITIAL_POLARITY = np.array([1.0, 1.0])
CONDITIONS = [(2.5, "Teal"), (0.5, "Blue")]
NATURE_PALETTES = {
    "Teal": ("#C5E3EB", "#8CCCCE", "#42B4B5", "#008F99", "#015D6E", "#003245"),
    "Blue": ("#C1E4FA", "#92C4E9", "#4C8FCB", "#0068A9", "#004586", "#002753"),
}

BASE = SimulationConfig(
    n_killers=8, n_targets=32, seed=0,
    killer_radius=10.0, target_radius=12.0,
    duration=10.0, max_dt=0.005, max_drift_displacement=1.0,
    initial_separation_factor=1.0,
    domain=DomainConfig(-300.0, 300.0, -300.0, 300.0, "periodic"),
    motility=MotilityConfig(
        killer_polarity_decay=2.5, killer_polarity_noise=0.2,
        killer_translation_noise=5.0, killer_speed=150.0,
        target_polarity_decay=2.5, target_polarity_noise=0.2,
        target_translation_noise=0.2, target_speed=50.0,
    ),
    mechanics=MechanicsConfig(
        repulsion_kk=250.0, repulsion_tt=250.0, repulsion_kt=250.0,
        adhesion_kt=25.0, binding_rate=3.0, unbinding_rate=0.5,
        capture_radius_factor=1.5, contact_tolerance_factor=0.05,
    ),
    cytotoxicity=CytotoxicityConfig(),
    output=OutputConfig(record_every=1),
)
BASE.validate()
initial_polarities = np.tile(INITIAL_POLARITY, (BASE.n_killers, 1))
assert INITIAL_POLARITY.shape == (2,) and np.isfinite(INITIAL_POLARITY).all()
assert np.linalg.norm(INITIAL_POLARITY) > 0, "Choose a nonzero initial polarity to show decay."


### 2. Simulate and retain polarity

Pass the initial vectors to `Simulation.run` and retain each cell's polarity
before discarding the snapshots. Check the starting vectors, time grid and finite
outputs. The expected mean is a reference for the stochastic process; individual
histories are not required to follow it.


In [ ]:
runs = {}
for alpha, _ in CONDITIONS:
    config = replace(BASE, motility=replace(BASE.motility, killer_polarity_decay=alpha))
    result = Simulation(config).run(initial_killer_polarities=initial_polarities)
    killers = result.snapshots.loc[result.snapshots.cell_type.eq("killer")].sort_values(
        ["step", "cell_id"]
    )
    times = killers.drop_duplicates("step").time.to_numpy()
    polarity = killers[["polarity_x", "polarity_y"]].to_numpy().reshape(
        -1, BASE.n_killers, 2
    )
    expected_mean = np.exp(-alpha * times[:, None]) * INITIAL_POLARITY

    assert polarity.shape == (len(times), BASE.n_killers, 2)
    assert np.all(np.diff(times) > 0)
    assert times[0] == 0 and np.isclose(times[-1], BASE.duration)
    assert np.isfinite(polarity).all()
    np.testing.assert_array_equal(polarity[0], initial_polarities)

    runs[alpha] = {"times": times, "polarity": polarity, "expected_mean": expected_mean}
    print(f"α = {alpha:g} min⁻¹: {BASE.n_killers} stochastic cell histories")
    del result, killers


### 3. Polarity components over time

Shades distinguish individual cells within each condition. The dashed curve shows
the theoretical ensemble mean. Both conditions use the same axis scales. The figure
is displayed here and saved as SVG, PNG and PDF in `figures/`.


In [ ]:
fig, axes = plt.subplots(
    2, 2, figsize=(8, 6.4), sharex=True, sharey=True,
    constrained_layout=True, dpi=300,
)
for col, (alpha, family) in enumerate(CONDITIONS):
    run = runs[alpha]
    times = run["times"]
    cmap = LinearSegmentedColormap.from_list(family, NATURE_PALETTES[family])
    colors = cmap(np.linspace(0.35, 0.9, BASE.n_killers))
    for component in range(2):
        ax = axes[component, col]
        for cell, color in enumerate(colors):
            ax.plot(times, run["polarity"][:, cell, component], color=color,
                    lw=1.2, label="Individual cell" if cell == 0 else None)
        ax.plot(times, run["expected_mean"][:, component], color="black",
                lw=1.8, ls="--", zorder=3, label="Expected mean")
        ax.axhline(0, color="black", lw=0.7, alpha=0.2)
        ax.set_xlim(0, BASE.duration)
        ax.tick_params(labelsize=16, length=7, width=1.25)
        ax.xaxis.set_major_locator(MaxNLocator(4))
        ax.yaxis.set_major_locator(MaxNLocator(3))
    axes[0, col].set_title(
        rf"$\alpha = {alpha:g}\,\mathrm{{min}}^{{-1}}$", fontsize=20, pad=12
    )

axes[0, 0].set_ylabel(r"Polarity $\mu_x$", fontsize=20)
axes[1, 0].set_ylabel(r"Polarity $\mu_y$", fontsize=20)
axes[0, 0].legend(loc="upper right", fontsize=12)
fig.supxlabel("Time (min)", fontsize=24, fontstyle="normal")

for extension in ("svg", "png", "pdf"):
    fig.savefig(FIGURE_DIR / f"polarity_decay_initial_condition.{extension}",
                dpi=300, bbox_inches="tight")
display(fig)
plt.close(fig)


In [ ]:
fig, axes = plt.subplots(
    1, 2, figsize=(8, 4.4), sharex=True, sharey=True,
    constrained_layout=True, dpi=300,
)

for ax, (alpha, family) in zip(axes, CONDITIONS):
    run = runs[alpha]
    times = run["times"]
    magnitude = np.linalg.norm(run["polarity"], axis=2)
    assert np.isfinite(magnitude).all()

    cmap = LinearSegmentedColormap.from_list(family, NATURE_PALETTES[family])
    colors = cmap(np.linspace(0.35, 0.9, magnitude.shape[1]))

    for cell, color in enumerate(colors):
        ax.plot(
            times, magnitude[:, cell], color=color, lw=1.2,
            label="Individual cells" if cell == 0 else None,
        )
        
    analytical = np.linalg.norm(INITIAL_POLARITY) * np.exp(-alpha * times)
    ax.plot(times, analytical, color="#D55E00", lw=2, ls="--", zorder=3, label="Deterministic reference")

    # ax.plot(
    #     times, magnitude.mean(axis=1),
    #     color="black", lw=1.8, ls="--", zorder=3,
    #     label="Mean magnitude",
    # )

    # ax.set_xlim(0, BASE.duration)
    ax.set_xlim(0, 6)
    ax.set_ylim(bottom=-0.1)
    ax.tick_params(labelsize=16, length=7, width=1.25)
    ax.xaxis.set_major_locator(MaxNLocator(4))
    ax.yaxis.set_major_locator(MaxNLocator(4))
    ax.set_title(
        rf"$\alpha = {alpha:g}\,\mathrm{{min}}^{{-1}}$",
        fontsize=20, pad=12,
    )

axes[0].set_ylabel(r"Polarity magnitude $\|\mu\|$", fontsize=20)
axes[0].legend(loc="upper right", fontsize=12)
axes[1].legend(loc="upper right", fontsize=12)
fig.supxlabel("Time (min)", fontsize=24, fontstyle="normal")

for extension in ("svg", "pdf"):
    fig.savefig(
        FIGURE_DIR / f"polarity_magnitude_initial_condition.{extension}",
        dpi=300, bbox_inches="tight",
        transparent=True
    )

display(fig)
plt.close(fig)